# Installing Dependencies

In [3]:
!pip install adapters
!pip install datasets scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# Importing Libraries

In [4]:
import os
import math
import itertools
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

import adapters
from adapters import AutoAdapterModel, AdapterConfig, AdapterTrainer
from adapters.composition import Stack

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

In [5]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Configuration

In [ ]:
BASE_MODEL = 'indolem/indobert-base-uncased'

SOURCE_LABELED_CSV     = 'datasets/id-coastsent_train.csv'   # source labeled
SOURCE_UNLABELED_CSV   = 'datasets/id-coastsent_train.csv'   # same file used for unlabeled domain
TARGET_UNLABELED_CSV   = 'datasets/lazada_train.csv'          # target unlabeled
TARGET_EVAL_CSV        = 'datasets/lazada_test.csv'           # target evaluation

# Labels 
LABEL2ID = {'negative': 0, 'positive': 1}
ID2LABEL = {0: 'negative', 1: 'positive'}

# Tokenization
MAX_LENGTH = 128
SEED       = 42

# Domain Adapter
DOMAIN_ADAPTER_NAME      = 'domain_adapter'
DOMAIN_ADAPTER_REDUCTION = 16          # adapter_dim=16 in friend's code
DOMAIN_EPOCHS            = 3
BATCH_SIZE               = 32         
DOMAIN_LR                = 5e-5        
WARMUP_RATIO             = 0.02     
WEIGHT_DECAY             = 0.01  
MAX_GRAD_NORM            = 1.0      

# MMD
MMD_KERNEL_MUL = 2.0
MMD_KERNEL_NUM = 5

# Task Adapter
TASK_ADAPTER_NAME      = 'task_adapter'
TASK_ADAPTER_REDUCTION = 16      
TASK_EPOCHS            = 3
TASK_LR                = 1e-4       
LABEL_SMOOTHING        = 0.05      

OUTPUT_DIR = './indobert_adapter_only'

# Dropout
DROPOUT = 0.0            

In [7]:
print('Configuration loaded.')
print(f'  Base model               : {BASE_MODEL}')
print(f'  Source labeled CSV       : {SOURCE_LABELED_CSV}')
print(f'  Source unlabeled CSV     : {SOURCE_UNLABELED_CSV}')
print(f'  Target unlabeled CSV     : {TARGET_UNLABELED_CSV}')
print(f'  Target eval CSV          : {TARGET_EVAL_CSV}')
print(f'  Batch size               : {BATCH_SIZE}')
print(f'  Domain adapter reduction : {DOMAIN_ADAPTER_REDUCTION}')
print(f'  Task adapter reduction   : {TASK_ADAPTER_REDUCTION}')
print(f'  Domain LR                : {DOMAIN_LR}')
print(f'  Task LR                  : {TASK_LR}')
print(f'  Warmup ratio             : {WARMUP_RATIO}')
print(f'  Label smoothing          : {LABEL_SMOOTHING}')

# Load Data

In [ ]:
# Label normalisation helpers
def _resolve_text_column(df):
    for col in ['content', 'reviewContent', 'text', 'review']:
        if col in df.columns: return col
    raise ValueError(f'No text column found. Got: {df.columns.tolist()}')

def _resolve_label_column(df):
    for col in ['label', 'sentiment', 'target']:
        if col in df.columns: return col
    raise ValueError(f'No label column found. Got: {df.columns.tolist()}')

def _normalize_labels(series):
    return series.astype(str).str.strip().str.lower().replace({
        'pos':     'positive', 'neg':     'negative',
        '1':       'positive', '0':       'negative',
        'positif': 'positive', 'negatif': 'negative',
        'true':    'positive', 'false':   'negative',
    })

def clean_dataframe(df, text_col, label_col=None):
    cols = [text_col] + ([label_col] if label_col else [])
    out  = df[cols].copy().dropna(subset=[text_col])
    out[text_col] = out[text_col].astype(str).str.strip()
    out  = out[out[text_col] != '']
    if label_col:
        out = out.dropna(subset=[label_col])
        out[label_col] = _normalize_labels(out[label_col])
        out = out[out[label_col].isin(LABEL2ID)]
    return out.reset_index(drop=True)

# Load source labeled data
source_df = pd.read_csv(SOURCE_LABELED_CSV)
TEXT_COL_SOURCE = _resolve_text_column(source_df)
LABEL_COL       = _resolve_label_column(source_df)
source_df = clean_dataframe(source_df, TEXT_COL_SOURCE, LABEL_COL)
source_df['label_id'] = source_df[LABEL_COL].map(LABEL2ID)

n_classes = source_df[LABEL_COL].nunique()
assert n_classes == 2, f'Expected 2 classes, got {n_classes}. Check label values: {source_df[LABEL_COL].unique()}'

print(f'Source dataset: {source_df.shape}')
print(source_df[LABEL_COL].value_counts())

In [ ]:
# Load target unlabeled data 
target_df = pd.read_csv(TARGET_UNLABELED_CSV)
TEXT_COL_TARGET = _resolve_text_column(target_df)
target_df = clean_dataframe(target_df, TEXT_COL_TARGET)
target_texts = target_df[TEXT_COL_TARGET].tolist()

print(f'Target dataset (unlabeled): {len(target_texts)} samples')
print(f'Sample source : {source_df[TEXT_COL_SOURCE].iloc[0]}')
print(f'Sample target : {target_texts[0]}')

# Training Setup

## Load Tokenizer and Base Model

In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model     = AutoAdapterModel.from_pretrained(BASE_MODEL)

adapters.init(model)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded  : {model.__class__.__name__}')
print(f'Total params  : {total_params:,}')


## MMD Loss Function

In [11]:
def gaussian_kernel(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    n_samples = source.size(0) + target.size(0)
    total     = torch.cat([source, target], dim=0)

    total0 = total.unsqueeze(0).expand(n_samples, n_samples, -1)
    total1 = total.unsqueeze(1).expand(n_samples, n_samples, -1)
    L2_distance = ((total0 - total1) ** 2).sum(2)

    if fix_sigma:
        bandwidth = fix_sigma
    else:
        bandwidth = torch.sum(L2_distance.detach()) / (n_samples ** 2 - n_samples)

    bandwidth /= kernel_mul ** (kernel_num // 2)
    bandwidth_list = [bandwidth * (kernel_mul ** i) for i in range(kernel_num)]

    kernel_val = [torch.exp(-L2_distance / bw) for bw in bandwidth_list]
    return sum(kernel_val)

In [12]:
def mmd_loss(source, target, kernel_mul=2.0, kernel_num=5, fix_sigma=None):
    batch_size = source.size(0)
    kernels    = gaussian_kernel(source, target, kernel_mul, kernel_num, fix_sigma)

    XX = kernels[:batch_size, :batch_size].mean()
    YY = kernels[batch_size:, batch_size:].mean()
    XY = kernels[:batch_size, batch_size:].mean()

    return XX + YY - 2 * XY

In [13]:
print('MMD loss defined.')
print(f'  Kernel multiplier : {MMD_KERNEL_MUL}')
print(f'  Number of kernels : {MMD_KERNEL_NUM}')

MMD loss defined.
  Kernel multiplier : 2.0
  Number of kernels : 5


##  Prepare DataLoaders for Domain Adapter

In [14]:
def tokenize_texts(texts, max_length=MAX_LENGTH):
    ds = Dataset.from_dict({'text': texts})
    ds = ds.map(
        lambda ex: tokenizer(
            ex['text'],
            truncation=True,
            max_length=max_length,
            padding=False
        ),
        batched=True,
        remove_columns=['text'],
        desc='Tokenizing'
    )
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask'])
    return ds

In [15]:
source_texts_domain = source_df[TEXT_COL_SOURCE].tolist()

source_domain_ds = tokenize_texts(source_texts_domain)
target_domain_ds = tokenize_texts(target_texts)

In [ ]:
collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors='pt')

# seed generators so shuffling is reproducible
g_src = torch.Generator(); g_src.manual_seed(SEED)
g_tgt = torch.Generator(); g_tgt.manual_seed(SEED + 1)

source_domain_loader = DataLoader(
    source_domain_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collator, generator=g_src
)
target_domain_loader = DataLoader(
    target_domain_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collator, generator=g_tgt
)

In [17]:
print(f'Source domain batches : {len(source_domain_loader)}')
print(f'Target domain batches : {len(target_domain_loader)}')

# Add Domain Adapter to Model

In [18]:
domain_adapter_config = AdapterConfig.load(
    'pfeiffer',
    reduction_factor=DOMAIN_ADAPTER_REDUCTION
)

model.add_adapter(DOMAIN_ADAPTER_NAME, config=domain_adapter_config)
model.train_adapter(DOMAIN_ADAPTER_NAME)
model.to(device)

In [19]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Domain adapter added : {DOMAIN_ADAPTER_NAME}')
print(f'Trainable params     : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [20]:
# Verifikasi semua parameter sudah di device yang sama
devices = set(p.device for p in model.parameters())
print(f'Model devices        : {devices}')  # harus cuma 1 device

# Model Training

## 1. Train Domain Adapter (MMD)

In [21]:
optimizer_domain = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=DOMAIN_LR,
    weight_decay=WEIGHT_DECAY,
)

In [22]:
model.train()

In [23]:
# Cek dulu struktur output model
with torch.no_grad():
    sample = next(iter(source_domain_loader))
    sample = {k: v.to(device) for k, v in sample.items()}
    test_out = model(
        input_ids=sample['input_ids'],
        attention_mask=sample['attention_mask'],
        output_hidden_states=True
    )
    print(f'Output type   : {type(test_out)}')
    print(f'Output keys   : {test_out.keys() if hasattr(test_out, "keys") else dir(test_out)}')
    if hasattr(test_out, 'hidden_states') and test_out.hidden_states is not None:
        print(f'Hidden states : {len(test_out.hidden_states)} layers, shape: {test_out.hidden_states[-1].shape}')

In [24]:
domain_losses = []
global_step   = 0

In [ ]:
# Cosine scheduler with warmup
steps_per_epoch_domain = max(len(source_domain_loader), len(target_domain_loader))
total_domain_steps     = DOMAIN_EPOCHS * steps_per_epoch_domain
warmup_domain_steps    = int(WARMUP_RATIO * total_domain_steps)

scheduler_domain = get_cosine_schedule_with_warmup(
    optimizer_domain,
    num_warmup_steps=warmup_domain_steps,
    num_training_steps=total_domain_steps,
)

print(f'Domain training: {DOMAIN_EPOCHS} epochs x {steps_per_epoch_domain} steps = {total_domain_steps} total')
print(f'Warmup steps: {warmup_domain_steps}')

In [25]:
epoch_pbar = tqdm(range(DOMAIN_EPOCHS), desc='Epochs', position=0)

In [ ]:
for epoch in epoch_pbar:
    epoch_loss = 0.0

    # Use max() + cycle() so ALL data from both domains is seen every epoch
    # (matches friend's train_domain_adapter approach)
    steps_this_epoch = max(len(source_domain_loader), len(target_domain_loader))
    src_iter = itertools.cycle(source_domain_loader)
    tgt_iter = itertools.cycle(target_domain_loader)

    step_pbar = tqdm(
        range(steps_this_epoch),
        desc=f'  Epoch {epoch+1}/{DOMAIN_EPOCHS}',
        position=1,
        leave=False
    )

    for step in step_pbar:
        src_batch = {k: v.to(device) for k, v in next(src_iter).items()}
        tgt_batch = {k: v.to(device) for k, v in next(tgt_iter).items()}

        src_outputs = model(
            input_ids=src_batch['input_ids'],
            attention_mask=src_batch['attention_mask'],
            output_hidden_states=True
        )
        tgt_outputs = model(
            input_ids=tgt_batch['input_ids'],
            attention_mask=tgt_batch['attention_mask'],
            output_hidden_states=True
        )

        # Mean pooling (matches friend's _mean_pool; richer than CLS alone) ──
        def mean_pool(hidden_states, attention_mask):
            last_h = hidden_states[-1]                        # (B, L, H)
            mask   = attention_mask.unsqueeze(-1).float()     # (B, L, 1)
            return (last_h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        src_feat = mean_pool(src_outputs.hidden_states, src_batch['attention_mask'])
        tgt_feat = mean_pool(tgt_outputs.hidden_states, tgt_batch['attention_mask'])

        loss = mmd_loss(src_feat, tgt_feat, MMD_KERNEL_MUL, MMD_KERNEL_NUM)

        optimizer_domain.zero_grad()
        loss.backward()
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()),
            MAX_GRAD_NORM
        )
        optimizer_domain.step()
        scheduler_domain.step()

        epoch_loss += loss.item()
        global_step += 1
        avg_loss     = epoch_loss / (step + 1)

        step_pbar.set_postfix({
            'MMD Loss': f'{avg_loss:.6f}',
            'step'    : f'{step+1}/{steps_this_epoch}'
        })

    step_pbar.close()

    avg_epoch_loss = epoch_loss / steps_this_epoch
    domain_losses.append(avg_epoch_loss)

    epoch_pbar.set_postfix({
        'avg MMD Loss': f'{avg_epoch_loss:.6f}',
        'epoch'       : f'{epoch+1}/{DOMAIN_EPOCHS}'
    })

    tqdm.write(f'Epoch {epoch+1}/{DOMAIN_EPOCHS} done — avg MMD loss: {avg_epoch_loss:.6f}')
    tqdm.write('-' * 60)

In [27]:
print(f'MMD loss per epoch: {[f"{l:.6f}" for l in domain_losses]}')

In [28]:
domain_adapter_dir = os.path.join(OUTPUT_DIR, 'domain_adapter')
os.makedirs(domain_adapter_dir, exist_ok=True)

model.save_adapter(domain_adapter_dir, DOMAIN_ADAPTER_NAME)

print(f'Domain adapter saved to: {domain_adapter_dir}')
for f in os.listdir(domain_adapter_dir):
    fpath   = os.path.join(domain_adapter_dir, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f:30s} ({size_kb:.1f} KB)')

In [ ]:
train_df, test_df = train_test_split(
    source_df,
    test_size=0.2,
    stratify=source_df['label_id'],
    random_state=SEED
)

# Class weights (matches friend's train_task_adapter) ─────────────────
counts   = source_df['label_id'].value_counts().sort_index()
total    = len(source_df)
class_weights = torch.tensor(
    [total / (2 * counts[i]) for i in range(2)],
    dtype=torch.float
).to(device)
print(f'Class weights: neg={class_weights[0]:.3f}  pos={class_weights[1]:.3f}')

In [30]:
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

In [31]:
def make_labeled_dataset(df, text_col):
    ds = Dataset.from_dict({
        'text'  : df[text_col].tolist(),
        'labels': df['label_id'].tolist()
    })
    ds = ds.map(
        lambda ex: tokenizer(
            ex['text'],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False
        ),
        batched=True,
        remove_columns=['text'],
        desc='Tokenizing'
    )
    return ds

In [32]:
train_dataset = make_labeled_dataset(train_df, TEXT_COL_SOURCE)
test_dataset  = make_labeled_dataset(test_df,  TEXT_COL_SOURCE)

print(f'Train dataset features: {train_dataset.features}')

Tokenizing:   0%|          | 0/47989 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/11998 [00:00<?, ? examples/s]

Train dataset features: {'labels': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


# Add Task Adapter (Stacked on Domain Adapter)

In [33]:
task_adapter_config = AdapterConfig.load(
    'pfeiffer',
    reduction_factor=TASK_ADAPTER_REDUCTION
)

model.add_adapter(TASK_ADAPTER_NAME, config=task_adapter_config)
model.add_classification_head(TASK_ADAPTER_NAME, num_labels=2, id2label=ID2LABEL)

# Stack: domain adapter (frozen) → task adapter (trainable)
model.active_adapters = Stack(DOMAIN_ADAPTER_NAME, TASK_ADAPTER_NAME)
model.train_adapter(TASK_ADAPTER_NAME)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Task adapter added   : {TASK_ADAPTER_NAME}')
print(f'Active stack         : {DOMAIN_ADAPTER_NAME} → {TASK_ADAPTER_NAME}')
print(f'Trainable params     : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

Task adapter added   : task_adapter
Active stack         : domain_adapter → task_adapter
Trainable params     : 2,078,786 / 113,563,445 (1.83%)


# Train Task Adapter

In [ ]:
task_training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, 'task_adapter_checkpoints'),
    num_train_epochs=TASK_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=TASK_LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,         
    label_smoothing_factor=LABEL_SMOOTHING, 
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    max_grad_norm=MAX_GRAD_NORM,     
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    report_to='none',
    seed=SEED
)

In [ ]:
# Weighted loss trainer 
class WeightedAdapterTrainer(AdapterTrainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device),
            label_smoothing=LABEL_SMOOTHING,
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [35]:
task_trainer = WeightedAdapterTrainer(
    class_weights=class_weights,
    model=model,
    args=task_training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

In [37]:
task_result = task_trainer.train()

In [38]:
print(f'Training loss : {task_result.training_loss:.4f}')
print(f'Training time : {task_result.metrics["train_runtime"]:.1f} s')

# Evaluate on Source Test Set

In [39]:
eval_results = task_trainer.evaluate()

print(f'\n  Accuracy  : {eval_results["eval_accuracy"]:.4f}')
print(f'  Precision : {eval_results["eval_precision"]:.4f}')
print(f'  Recall    : {eval_results["eval_recall"]:.4f}')
print(f'  F1 Score  : {eval_results["eval_f1"]:.4f}')
print(f'  Loss      : {eval_results["eval_loss"]:.4f}')

In [40]:
predictions = task_trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print('\nDetailed Classification Report:')
print('=' * 60)
print(classification_report(
    true_labels, pred_labels,
    target_names=['negative', 'positive'],
    digits=4
))

# Evaluate on Mixed Test Set (Source + Target)

In [41]:
mixed_df = pd.read_csv('mixed_test.csv')
mixed_df = mixed_df.dropna(subset=['content', 'label'])
mixed_df['label_id'] = mixed_df['label'].map(LABEL2ID)

print(f'Mixed test set: {mixed_df.shape}')
print(mixed_df['origin'].value_counts())

In [42]:
mixed_dataset     = make_labeled_dataset(mixed_df, 'content')
mixed_predictions = task_trainer.predict(mixed_dataset)
mixed_pred_labels = np.argmax(mixed_predictions.predictions, axis=1)
mixed_true_labels = mixed_predictions.label_ids

mixed_df['predicted_label_id'] = mixed_pred_labels
mixed_df['predicted_label']    = mixed_df['predicted_label_id'].map(ID2LABEL)

overall_acc = accuracy_score(mixed_true_labels, mixed_pred_labels)
print(f'\nOVERALL ACCURACY: {overall_acc:.4f} ({overall_acc*100:.2f}%)')
print('=' * 60)

for origin in mixed_df['origin'].unique():
    mask   = mixed_df['origin'] == origin
    o_true = mixed_df.loc[mask, 'label_id'].values
    o_pred = mixed_df.loc[mask, 'predicted_label_id'].values

    o_acc = accuracy_score(o_true, o_pred)
    o_prec, o_rec, o_f1, _ = precision_recall_fscore_support(
        o_true, o_pred, average='weighted'
    )

    print(f'\n{origin.upper()} domain ({mask.sum()} samples):')
    print(f'  Accuracy  : {o_acc:.4f}')
    print(f'  Precision : {o_prec:.4f}')
    print(f'  Recall    : {o_rec:.4f}')
    print(f'  F1        : {o_f1:.4f}')
    print(classification_report(
        o_true, o_pred,
        target_names=['negative', 'positive'],
        digits=4
    ))

In [43]:
# Save Final Model

In [44]:
final_dir = os.path.join(OUTPUT_DIR, 'final')
os.makedirs(final_dir, exist_ok=True)

model.save_adapter(os.path.join(final_dir, 'domain_adapter'), DOMAIN_ADAPTER_NAME)
model.save_adapter(os.path.join(final_dir, 'task_adapter'),   TASK_ADAPTER_NAME)
tokenizer.save_pretrained(final_dir)

print(f'Saved to: {final_dir}')
print('\nTo load later:')
print(f'  model = AutoAdapterModel.from_pretrained("{BASE_MODEL}")')
print(f'  model.load_adapter("{final_dir}/domain_adapter")')
print(f'  model.load_adapter("{final_dir}/task_adapter")')
print(f'  model.active_adapters = Stack("domain_adapter", "task_adapter")')

# Quick Inference Test

In [45]:
model.eval()

In [53]:
test_samples = [
    # App / Tech
    'Aplikasinya sangat membantu dan mudah digunakan!',
    'Aplikasi sering error dan lambat. Tolong diperbaiki.',

    # E-commerce
    'Pengiriman cepat, barang sesuai deskripsi. Sangat puas!',
    'Barang datang rusak dan tidak sesuai ekspektasi.',

    # Food / Restaurant
    'Makanannya enak banget, bumbunya pas!',
    'Pelayanan lama dan makanan datang dingin.',

    # Transport
    'Driver sangat ramah dan tepat waktu.',
    'Driver tidak sopan dan rutenya muter-muter.',

    # Hotel / Travel
    'Kamarnya bersih dan nyaman, sangat rekomendasi!',
    'AC tidak dingin dan kamar kurang terawat.',

    # Education
    'Materinya mudah dipahami dan sangat membantu belajar.',
    'Penjelasan kurang jelas dan membingungkan.',

    # Neutral / mixed
    'Pelayanannya lumayan, masih bisa diimprove',
    'Cukup oke, tapi ada beberapa kekurangan kecil'
]

In [54]:
for text in test_samples:
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs      = torch.softmax(outputs.logits, dim=-1)
    pred_id    = probs.argmax().item()
    confidence = probs[0][pred_id].item()

    print(f'Text       : {text}')
    print(f'Prediction : {ID2LABEL[pred_id].upper()} (confidence: {confidence:.4f})')
    print('-' * 60)

In [48]:
!zip -r /content/adapter_only.zip /content/indobert_adapter_only